In [ ]:
import warnings
import sleap
import numpy as np

# Print some info:
sleap.versions()
sleap.system_summary()

# Load files

In [ ]:
labels_ch = sleap.load_file("full_pose_ch_predictions.slp")
labels_ap = sleap.load_file("full_pose_ap_predictions.slp")
labels_ap_full = labels_ap.copy()
if labels_ch.video.filename != labels_ap.video.filename:
    raise ValueError("Video files don't match!")

In [ ]:
nodes_to_remove = ['nose', 'head', 'right_ear', 'left_ear', 'spine1', 'spine3', 'spine4']
for node in nodes_to_remove:
    labels_ap.skeleton.delete_node(node)

# Match instances and create full pose .slp file with ID

In [ ]:
# Match frames and instances
frame_pairs = sleap.nn.evals.find_frame_pairs(labels_ch, labels_ap, user_labels_only=False)
matched_instances = sleap.nn.evals.match_frame_pairs(frame_pairs, stddev = 0.05, scale = 2.5, threshold=0, user_labels_only=False)

j = 0
instances = []
frame_idx = matched_instances[0][0][0].frame_idx
full_pose_frame_data = labels_ap_full.find(labels_ap_full.video, frame_idx)[0]
candidate_points = [instance.points_array[5] for instance in full_pose_frame_data.instances]
lfs=[]
# Loop over instances
for j in range(len(matched_instances[0])):
    # Find the index of the mouse who was matched to the current identity we are considering (matched_instance[0])
    # This is done in order to get back the full pose (labels_ap only has spine2)
    instance_idx = np.where(candidate_points == matched_instances[0][j][1].points_array[0])[0][0]
    instances.append(
        sleap.PredictedInstance(
            skeleton=labels_ap_full.skeleton,
            track=matched_instances[0][j][0].track,
            points=full_pose_frame_data[instance_idx].get_points_array(copy=False, full=True),
        )
    )
    # Update the frame if we are at the end of the current frame
    if j < len(matched_instances[0])-1 and matched_instances[0][j+1][0].frame_idx != matched_instances[0][j][0].frame_idx:
        instances = []
        frame_idx = matched_instances[0][j+1][0].frame_idx
        full_pose_frame_data = labels_ap_full.find(labels_ap_full.video, frame_idx)[0]
        candidate_points = [instance.points_array[5] for instance in full_pose_frame_data.instances]
        lf = sleap.instance.LabeledFrame(
            video=labels_ap.video,
            frame_idx=frame_idx,
            instances=instances,
        )
        lfs.append(lf)
labels_ap_id = sleap.Labels(labeled_frames=lfs)
sleap.Labels.save_file(labels_ap_id, "full_pose_predictions.slp")